In [ ]:
from prysm.interferogram import (
    abc_psd,
    render_synthetic_surface
)

from prysm.coordinates import make_xy_grid, cart_to_polar
from prysm.propagation import focus_fixed_sampling
from prysm.geometry import circle
from prysm.x.dm import DM

from poi.influence_funcs import gaussian_influence_function

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from matplotlib.patches import Circle

# Set up a PSD wavefront error

In [ ]:
nu = np.arange(1,1000,1)

In [ ]:
psd = abc_psd(nu, a=1, b=10, c=1)

plt.figure()
plt.plot(nu, psd)
plt.xscale('log')
plt.yscale('log')
plt.ylabel('PSD')
plt.xlabel('Spatial Frequency')
plt.show()

In [ ]:
# Just picked ABC values until it looked about right
Npup = 256
x, y, z = render_synthetic_surface(10, Npup, a=5e4, b=1/1000, c=3)

In [ ]:
plt.figure()
plt.title('Synthetic WFE')
plt.imshow(z, cmap='RdBu_r')
plt.colorbar(label='microns WFE')
plt.show()

# 2. Set up a diffraction model

In [ ]:
x, y = make_xy_grid(Npup, diameter=2)
r, t = cart_to_polar(x, y)

A = circle(0.9, r)
LS = circle(0.8,r)
plt.figure()
plt.imshow(A, cmap='gray')
plt.title('Telescope Aperture')
plt.show()

In [ ]:
# create pupil function and propagate
WVL = 1 # micron
EFL = 100 # mm
Dpup = 10 # mm
dx_img = 1 # microns
wvfnt = A * np.exp(1j * 2 * np.pi / WVL * z/2) # with error

focal = focus_fixed_sampling(wvfnt,
                             input_dx=Dpup / wvfnt.shape[0], # mm pupil
                             prop_dist=EFL, # mm focal length
                             wavelength=WVL, # microns
                             output_dx=dx_img, # microns
                             output_samples=wvfnt.shape[0])

focal_intensity = np.abs(focal)**2

In [ ]:
plt.title('PSF Before Correction')
plt.imshow(focal_intensity, cmap='inferno', norm=LogNorm())

In [ ]:
# create a dark hole
um_to_LD = 1e-6 / (EFL * 1e-3) * (Dpup * 1e-3) / (WVL * 1e-6)
print(um_to_LD)
u, v = make_xy_grid(focal.shape, dx=dx_img * um_to_LD)
rho, psi = cart_to_polar(u, v)

dark_hole = np.zeros_like(u)
dark_hole[rho < 5] = 1
dark_hole[rho < 3] = 0
dark_hole[u < 3] = 0

In [ ]:
masked_intensity = dark_hole * focal_intensity
plt.title('Control Region')
plt.imshow(masked_intensity, cmap='inferno', norm=LogNorm(), origin='lower')

plt.colorbar()

In [ ]:
yind, xind = np.unravel_index(np.argmax(masked_intensity), masked_intensity.shape)

In [ ]:
fig, ax = plt.subplots()
plt.title('Sensing Brightest Speckle')
im = ax.imshow(masked_intensity, cmap='inferno', norm=LogNorm(vmin=1e-2))
fig.colorbar(im)

circ = Circle((xind, yind), radius=5, color='cyan', fill=False, linewidth=2)
ax.add_patch(circ)
plt.show()


In [ ]:
# Set up DM
nact = 11 # number of actuators
act_pitch = 20 # mm?
samples_per_act = 21# chosen to over-fill illuminated aperture
sampling_pitch = act_pitch / samples_per_act

xdm, ydm = make_xy_grid(Npup, dx=sampling_pitch)
rhodm, _ = cart_to_polar(xdm, ydm)
influence_func = gaussian_influence_function(rho, sampling_pitch)

In [ ]:
plt.title("Gaussian Influence Function")
plt.imshow(influence_func)
plt.colorbar()
plt.show()

In [ ]:
dm = DM(influence_func, Nout=Npup, Nact=nact, sep=samples_per_act,
        shift=(samples_per_act, samples_per_act))

In [ ]:
# show DM footprint
dm.actuators[:] = 1

plt.imshow(dm.render())

# Generate map of locations, assume rotational symmetry

In [ ]:
# frequency = np.linspace(0, nact/2)

# for f in frequency:

from poi.modes import fourier_modes_sequence

modes = fourier_modes_sequence(nact)

# Create a Coronagraph Forward Model

In [ ]:
from prysm.propagation import unfocus_fixed_sampling

# create a dark hole
um_to_LD = 1e-6 / (EFL * 1e-3) * (Dpup * 1e-3) / (WVL * 1e-6)
print(um_to_LD)
u, v = make_xy_grid(focal.shape, dx=dx_img * um_to_LD)
rho, psi = cart_to_polar(u, v)

dark_hole = np.zeros_like(u)
IWA = 3.5
OWA = 5
dark_hole[rho < OWA] = 1
dark_hole[rho < IWA] = 0
dark_hole[psi > np.pi / 4] = 0
dark_hole[psi < -np.pi / 4] = 0

fpm = np.ones_like(u)
fpm[rho < 3] = 0


masked_intensity = dark_hole * focal_intensity
plt.title('Control Region')
plt.imshow(masked_intensity, cmap='inferno', norm=LogNorm(), origin='lower')

plt.colorbar()


def fwd(pupil_wfe=np.ones_like(wvfnt)):
    focal = focus_fixed_sampling(wvfnt * pupil_wfe * A,
                                input_dx=Dpup / wvfnt.shape[0], # mm pupil
                                prop_dist=EFL, # mm focal length
                                wavelength=WVL, # microns
                                output_dx=dx_img, # microns
                                output_samples=wvfnt.shape[0])
    
    focal *= fpm

    lyot = unfocus_fixed_sampling(focal,
                                input_dx=dx_img, # um pupil
                                prop_dist=EFL, # mm focal length
                                wavelength=WVL, # microns
                                output_dx=Dpup / wvfnt.shape[0], # mm
                                output_samples=wvfnt.shape[0])
    
    lyot *= LS
    
    focal = focus_fixed_sampling(lyot,
                                input_dx=Dpup / wvfnt.shape[0], # mm pupil
                                prop_dist=EFL, # mm focal length
                                wavelength=WVL, # microns
                                output_dx=dx_img, # microns
                                output_samples=wvfnt.shape[0])
    
    return focal

In [ ]:
from matplotlib.patches import Circle

for i, m in enumerate(modes):
    plt.figure()
    plt.subplot(121)
    plt.title(f"mode = {i+1}")
    plt.imshow(m, cmap="coolwarm")

    # image formulation
    dm.actuators[:] = m
    efield = fwd(1j * dm.render(wfe=True)) / 100
    im = np.abs(efield)**2
    plt.subplot(122)
    plt.imshow(np.log10(im), cmap="inferno", vmax=0, vmin=-7)
    ax = plt.gca()
    fpm_region = Circle((128, 128), 3 / um_to_LD, color="k", alpha=0.5)
    ax.add_patch(fpm_region)
    plt.show()

In [ ]:
def create_annular_focal_plane_mask(npsf, psf_pixelscale, 
                                    inner_radius, outer_radius, 
                                    edge=None,
                                    shift=(0,0), 
                                    rotation=0,
                                    plot=False):
    
    x = (np.linspace(-npsf/2, npsf/2-1, npsf) + 1/2)*psf_pixelscale
    x,y = np.meshgrid(x,x)
    r = np.hypot(x, y)
    mask = (r < outer_radius) * (r > inner_radius)

    if edge is not None:
        mask *= (x > edge)
        
    return mask

def fourier_modes_sequence(Nact, Nimg, psf_pixelscale_lamD, iwa, owa,
                           fourier_sampling=1, which="cos", return_frequencies_sampled=False):

    nfg = int(Nimg * psf_pixelscale_lamD / fourier_sampling)

    if nfg%2 == 1:
        nfg += 1

    y_freq, x_freq = np.indices((nfg, nfg)) - nfg // 2 + 0.5
    y_freq *= fourier_sampling
    x_freq *= fourier_sampling

    fourier_mask = create_annular_focal_plane_mask(nfg, fourier_sampling, iwa-fourier_sampling, owa+fourier_sampling, edge=iwa-fourier_sampling)
    

    y_dm, x_dm = np.indices((Nact, Nact)) - Nact // 2 + 0.5

    sampled_frequencies = np.array([x_freq[fourier_mask], y_freq[fourier_mask]]).T

    fourier_modes = []
    for i, (fx, fy) in enumerate(zip(x_freq[fourier_mask], y_freq[fourier_mask])):

        if which=='both' or which=='cos':
            fourier_modes.append(np.cos(2 * np.pi * (fx*x_dm + fy*y_dm)/Nact) )
        if which=='both' or which=='sin':
            fourier_modes.append(np.sin(2 * np.pi * (fx*x_dm + fy*y_dm)/Nact) )
    
    if return_frequencies_sampled:
        return np.array(fourier_modes), sampled_frequencies
    else:
        return np.array(fourier_modes)

In [ ]:
modes = fourier_modes_sequence(nact, 256, um_to_LD, iwa=IWA, owa=OWA)

for i, m in enumerate(modes):
    plt.figure()
    plt.subplot(121)
    plt.title(f"mode = {i+1}")
    plt.imshow(m, cmap="coolwarm")

    # image formulation
    dm.actuators[:] = m
    efield = fwd(1j * dm.render(wfe=True)) / 100
    im = np.abs(efield)**2
    plt.subplot(122)
    plt.imshow(np.log10(im), cmap="inferno", vmax=0, vmin=-7)
    ax = plt.gca()
    fpm_region = Circle((128, 128), 3 / um_to_LD, color="k", alpha=0.5)
    dh_owa = Circle((128, 128), OWA / um_to_LD, facecolor="None", edgecolor="w")
    dh_iwa = Circle((128, 128), IWA / um_to_LD, facecolor="None", edgecolor="w")
    ax.add_patch(fpm_region)
    ax.add_patch(dh_iwa)
    ax.add_patch(dh_owa)
    plt.show()